# How Runnables solve the problem

In the previous example, prompt formatting and model invocation used unrelated method names: `format(...)` and `predict(...)`. That forces us to know the internal API of every component before we can connect them.

Here, each component follows the same small contract: it inherits from `Runnable` and implements `invoke(input_data)`. The input and output are still allowed to be different types—this example intentionally passes a dictionary to the prompt template, a string to the model, and a response dictionary to the parser. The common `invoke(...)` method standardizes **how a step is called**, while each step remains responsible for its own transformation.

This is the key idea behind a runnable pipeline: each step receives the previous step's output and returns the value needed by the next step.

In [29]:
import random
from abc import ABC, abstractmethod

In [30]:
class Runnable(ABC):
    @abstractmethod
    def invoke(self, input_data):
        pass

In [31]:
class NakliLLM(Runnable):
    def __init__(self, api_key=None):
        self.api_key = api_key

    def invoke(self, input_data):         # 1st way to implement invoke method
        return self.predict(input_data)

    def invoke1(self, prompt):        # 2nd way to implement invoke method
        responses = [
            f"I love {prompt}!",
            f"{prompt} is amazing!",
            f"I can't get enough of {prompt}!",
        ]
        return {"response" :  {random.choice(responses)}}

    def predict(self, prompt):    # by using second way , we can print warning that, this method is deprecated and will be removed in future versions. Instead, use invoke1 method.
        responses = [
            f"I love {prompt}!",
            f"{prompt} is amazing!",
            f"I can't get enough of {prompt}!",
        ]
        return {"response" :  {random.choice(responses)}}


In [32]:
class NakliPromptTemplate(Runnable):
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def invoke(self, input_data):  #1st way to implement invoke method
        return self.format(input_data)

    def invoke1(self, input_dict):  #2nd way to implement invoke method, we can print warning that, this method is deprecated and will be removed in future versions. Instead, use invoke method.
        return self.ftemplate.format(**input_dict)

    def format(self, input_dict):
        return self.template.format(**input_dict)


In [33]:
class NakliStrOutputParser(Runnable):
    def __init__(self):
        pass

    def invoke(self, input_data):
        return input_data['response'] # Assuming the input_data is a dictionary with a 'response' key

In [34]:
model = NakliLLM(api_key="fake_api_key")

In [35]:
template = NakliPromptTemplate(
    template="Create a {trip_length}-day itinerary for {destination}. The traveler enjoys {interests}. Include one practical tip for each day.",
    input_variables=["trip_length", "destination", "interests"]
)

In [36]:
prompt = template.invoke({"trip_length": 3, "destination": "Kyoto", "interests": "temples, local food, and quiet neighborhoods"})
print(prompt)

Create a 3-day itinerary for Kyoto. The traveler enjoys temples, local food, and quiet neighborhoods. Include one practical tip for each day.


In [37]:
result = model.invoke(prompt)
print(result)

{'response': {"I can't get enough of Create a 3-day itinerary for Kyoto. The traveler enjoys temples, local food, and quiet neighborhoods. Include one practical tip for each day.!"}}


In [38]:
# Take a look at the code above. We have defined two classes, NakliLLM and NakliPromptTemplate, both of which inherit from the Runnable abstract base class. Each class implements the invoke method, which allows us to call the model and format the prompt using a unified interface.

## Reading the first pipeline

The input moves through the components like this:

`{"trip_length": 3, "destination": "Kyoto", "interests": "temples, local food, and quiet neighborhoods"}` → **prompt template** → an itinerary request → **model** → `{"response": ...}`

The template and model do not need to know each other's method names. The caller only needs to know that both support `invoke(...)`. This makes it possible to replace one component with another Runnable without rewriting the calling code.

Now let's begin real fun part. We will create a simple LLM and a prompt template, both of which will inherit from the Runnable abstract base class. This will allow us to invoke them using a unified interface.That is chaining them together seamlessly.

In [39]:
class RunnableConnector(Runnable):
    def __init__(self, runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_data):
        for runnable in self.runnable_list:
            input_data = runnable.invoke(input_data)
        return input_data

In [40]:
# now Let's create one model, two prompt templates, and a RunnableConnector to chain them together. We will then invoke the RunnableConnector with an input dictionary, which will be passed through the chain of runnables.

## Why `RunnableConnector` is useful

`RunnableConnector` is a small example of composition. It stores an ordered list of Runnables and repeatedly calls `invoke(...)`, replacing the current value with each step's output.

Without this connector, we would write one manual call for every step and keep track of every intermediate variable. With it, the workflow can be treated as one Runnable: the caller supplies one input dictionary and receives the final result. Adding an output parser is also just adding another Runnable to the list; the parser receives the model's response dictionary and extracts the final value.

In [41]:
llm = NakliLLM(api_key="fake_api_key")

In [42]:
template1 = NakliPromptTemplate(
    template="Plan a {days}-day trip to {destination}. Focus on {travel_style}. Suggest a realistic order for the activities.",
    input_variables=["days", "destination", "travel_style"]
)

In [43]:
template2 = NakliPromptTemplate(
    template="Plan a {days}-day trip to {destination}. Focus on {travel_style}. Suggest a realistic order for the activities.",
    input_variables=["days", "destination", "travel_style"]
)

In [44]:
runnable_connector = RunnableConnector([template1, llm])

In [45]:
result = runnable_connector.invoke({"days": 3, "destination": "Kyoto", "travel_style": "temples, local food, and quiet neighborhoods"})
print(result)

{'response': {'Plan a 3-day trip to Kyoto. Focus on temples, local food, and quiet neighborhoods. Suggest a realistic order for the activities. is amazing!'}}


In [46]:
strOutput_parser = NakliStrOutputParser()

In [47]:
runnable_connector = RunnableConnector([template1, llm, strOutput_parser])


In [48]:
result = runnable_connector.invoke({"days": 3, "destination": "Kyoto", "travel_style": "temples, local food, and quiet neighborhoods"})

print(result)


{'Plan a 3-day trip to Kyoto. Focus on temples, local food, and quiet neighborhoods. Suggest a realistic order for the activities. is amazing!'}


## A more realistic multi-step scenario

A common application may need to:

1. Build a prompt from user data.
2. Ask the model to create content.
3. Summarize or transform that content.
4. Parse the result into the format used by the application.

Each stage has a different responsibility and may produce a different data shape. Runnables make the execution pattern consistent across all stages, so the complete workflow can be composed from smaller, independently understandable pieces. The examples below show this idea with a Kyoto itinerary chain followed by a traveler checklist chain.

## Important: a common interface does not remove data contracts

Runnables standardize the call (`invoke(...)`), not the meaning of the data. A prompt template still requires the keys named in its template, and the next template must accept the previous step's output.

For example, a template containing `{destination}` cannot receive a response that only contains `{response}`. If the keys do not line up, Python raises a `KeyError`. That error is useful: it shows exactly where the pipeline's input/output contract is mismatched. In a real chain, we would make the first template produce the fields required by the next template, or add a small Runnable transformation between them.

In [49]:
template1 = NakliPromptTemplate(
    template="Create a relaxed weekend itinerary for {destination}. Include local food, one cultural activity, and free time.",
    input_variables=["destination"]
)

In [50]:
template2 = NakliPromptTemplate(
    template= "Turn this itinerary into a short checklist for the traveler: {response}",
    input_variables=["response"]
)

In [51]:
llm = NakliLLM(api_key="fake_api_key")

In [52]:
parser = NakliStrOutputParser()

In [53]:
chain1 = RunnableConnector([template1, llm])

In [54]:
chain2 = RunnableConnector([template2, llm, parser])

In [55]:
final_chain = RunnableConnector([chain1, chain2])

In [56]:
result = final_chain.invoke({"destination": "Kyoto"})
print(result)

{'I can\'t get enough of Turn this itinerary into a short checklist for the traveler: {"I can\'t get enough of Create a relaxed weekend itinerary for Kyoto. Include local food, one cultural activity, and free time.!"}!'}
